# Class 07 · Manipulating and Visualizing Data (R)

Live-coding companion to the Week 4, Class A slides. Run the cells from top to bottom; each step builds on the previous one and ends with a saved figure, its summary table and the record of excluded rows.

## Part 1 · Tables and the meaning of a row

### Start with the meaning of a row

A table operation changes the available evidence. Our game example has one record per player–game pair. A second table supplies a genre and an achievement total for each listed game.

In [ ]:
library(dplyr)
library(ggplot2)

> Run the examples in order, in a folder where output files can be saved.

### A tibble extends a data frame

- A tibble inherits from `data.frame`, with its own printing and subsetting.
- `tb[, "x"]` keeps a table; `tb[["x"]]` extracts a vector.
- In pandas, `tb[["x"]]` is a DataFrame and `tb["x"]` is a Series.

In [ ]:
tb <- tibble(x = 1:3, y = c("a", "b", "c"))
df <- as.data.frame(tb)
print(class(tb))
print(class(df[, "x"]))
print(class(tb[, "x"]))
print(class(tb[["x"]]))

> Ask: which of these is still a table?

### The data: eight records and a game lookup

- Dan has one unknown hours value.
- Tunic appears in the play records but has no lookup entry.
- Stardew appears only in the lookup table.

Keep the original tables. Derived tables get new names.

In [ ]:
# Player-game records.
plays <- tibble(
  player = c("ana", "ana", "ben", "ben",
             "cara", "cara", "dan", "dan"),
  game = c("Portal", "Celeste", "Portal",
           "Hades", "Celeste", "Hades",
           "Portal", "Tunic"),
  hours = c(12.5, 30, 8, 45.5, 22, 17.5, NA, 6),
  achieved = c(9L, 14L, 4L, 25L, 11L, 8L,
               2L, 3L))

In [ ]:
# Game lookup.
games <- tibble(
  game = c("Portal", "Celeste", "Hades",
           "Stardew"),
  genre = c("puzzle", "platformer",
            "roguelike", "farming"),
  total = c(15L, 30L, 49L, 40L))
print(plays)
print(games)

> Each game key is unique in the lookup.

## Part 2 · Six table operations

### Select records

Conditions in `filter` are combined with AND; pandas keeps the original index labels.

In [ ]:
print(filter(plays, hours > 20))
print(filter(plays, hours > 20,
             player != "ana"))
print(filter(plays,
             game %in% c("Hades", "Tunic")))

> Predict first: which rows have more than 20 hours? (Ana–Celeste, Ben–Hades, Cara–Celeste)

### Make the missing-value rule explicit

- R comparisons with `NA` give an unknown result; `filter` keeps only `TRUE`.
- For the float `NaN` in pandas, `NaN > 20` is false but `NaN != 20` is true.
- Write the policy into the code instead of inheriting a default.

In [ ]:
print(nrow(filter(plays, hours > 20)))   # 3
print(nrow(filter(plays, hours != 20)))  # 7
base_plays <- as.data.frame(plays)
print(nrow(
  base_plays[base_plays$hours > 20, ]))  # 4
known <- filter(plays, !is.na(hours),
                hours != 20)
print(nrow(known))  # 7, explicit NA policy

> The explicit known-hours selection gives seven records in both languages.

### Select and rename variables

A label range includes both endpoints. An ordinary Python positional slice excludes its stop.

In [ ]:
print(select(plays, player, game, hours))
print(select(plays, -achieved))
print(select(plays, player:hours))
print(select(plays, where(is.numeric)))
print(rename(plays, hrs = hours))

### Order records

Missing hours appear last. Add ordering columns when ties matter.

In [ ]:
print(arrange(plays, desc(hours)))
print(arrange(plays, player, desc(achieved)))

### Derive a rate

Rate = achievements per known positive hour. Missing or non-positive hours give a missing rate and band.

In [ ]:
rated <- mutate(plays,
  denominator = if_else(hours > 0, hours,
                        NA_real_),
  rate = achieved / denominator,
  band = if_else(rate > 0.5, "hi", "lo"))
print(select(rated, player, game, rate, band))

> Ask: what should Dan–Portal's rate be?

### Report the summary denominator

Say how many values a summary uses.

In [ ]:
print(mean(plays$hours))  # NA
overview <- summarise(plays,
  records = n(),
  observed_hours = sum(!is.na(hours)),
  mean_hours = mean(hours, na.rm = TRUE),
  achievements = sum(achieved))
print(overview)
# 8 records, 7 observed, 20.214286 h, 76

> Four players, eight records, seven observed hours; the mean describes those seven.

## Part 3 · Grouped calculations

### Summarize each player

Choose groups, calculate one summary per group, then drop the grouping. In R, `x |> f()` passes `x` as the first argument of `f`; a pandas method chain passes the table to the next method.

In [ ]:
by_player <- plays |>
  group_by(player) |>
  summarise(records = n(),
    observed_hours = sum(!is.na(hours)),
    known_hours = sum(hours, na.rm = TRUE),
    achievements = sum(achieved),
    .groups = "drop")
print(by_player)

> Dan's known hours (6) are a partial observed sum: his Portal hours are unknown.

### Keep the original record count

A grouped mutate or transform keeps every record; a summary collapses them.

In [ ]:
shares <- plays |>
  group_by(player) |>
  mutate(share_known_hours =
    hours / sum(hours, na.rm = TRUE)) |>
  ungroup()
print(select(shares, player, game,
             share_known_hours))

> Eight records come back, not four.

### Control group order

In [ ]:
dan_first <- arrange(plays, desc(player))
print(summarise(dan_first,
  achievements = sum(achieved), .by = player))
# First appearance: dan, cara, ben, ana.

> First-appearance groups: dan, cara, ben, ana. Default grouped summaries sort keys.

### Retain an unobserved category

Eve is a possible category with no recorded rows. Absence from these records does not prove zero activity.

In [ ]:
levels_player <- c("ana", "ben", "cara",
                   "dan", "eve")
p <- mutate(plays,
  player = factor(player,
                  levels = levels_player))
empty_groups <- p |>
  group_by(player, .drop = FALSE) |>
  summarise(records = n(), .groups = "drop")
print(empty_groups)  # eve has 0 recorded rows

### Compose the six operations

The mean gives every eligible record equal weight. It is not total achievements divided by total hours.

In [ ]:
player_rates <- plays |>
  filter(!is.na(hours), hours > 0) |>
  mutate(rate = achieved / hours) |>
  group_by(player) |>
  summarise(records = n(),
            mean_rate = mean(rate),
            .groups = "drop") |>
  arrange(desc(mean_rate)) |>
  select(player, records, mean_rate)
print(player_rates)

## Part 4 · Combining tables

### Append compatible records

Appending rows aligns column names. Appending does not change the original lookup.

In [ ]:
extra <- tibble(game = "Tunic",
                genre = "puzzle", total = 42L)
extended <- bind_rows(games, extra)
print(extended)  # games itself still has 4 rows
a <- tibble(x = 1)
b <- tibble(y = 2)
print(bind_rows(a, b))  # missing cols become NA

> Check column names, units and types before binding.

### Check row correspondence

Adding columns by position needs an established correspondence between rows.

In [ ]:
x <- tibble(id = c("a", "b"), value = c(2, 3))
y <- tibble(id = c("a", "b"),
            label = c("low", "high"))
stopifnot(identical(x$id, y$id))
print(bind_cols(x, select(y, label)))

### Validate the lookup relationship

Check complete keys and a unique lookup key before the join. Duplicate lookup keys would create extra matches.

In [ ]:
stopifnot(!anyNA(plays$game),
          !anyNA(games$game))
stopifnot(!anyDuplicated(games$game))
joined <- left_join(plays, games, by = "game",
  relationship = "many-to-one",
  na_matches = "never")
stopifnot(nrow(joined) == nrow(plays))
print(joined)

> The left join keeps all eight play records.

### Compare the four join counts

In [ ]:
print(nrow(joined))                   # 8
print(nrow(inner_join(plays, games,
                      by = "game")))  # 7
print(nrow(right_join(plays, games,
                      by = "game")))  # 8
print(nrow(full_join(plays, games,
                     by = "game")))   # 9

> Predict first: inner 7, left 8, right 8, full 9. Why?

## Part 5 · Visualization

### Load the Iris measurements

The figure should answer a question already defined by the table. We revisit Iris from Week 3.

In [ ]:
d <- as_tibble(iris)
print(nrow(d))                 # 150 flowers
print(mean(d$Sepal.Width))     # 3.057333 cm

> 150 flowers; mean sepal width 3.057333 cm.

### Save the scatter plot

Select the output file device explicitly, draw, then close it.

In [ ]:
png("class07-iris-base.png", width = 1200,
    height = 800, res = 150)
plot(d$Sepal.Length, d$Sepal.Width, pch = 1,
     xlab = "Sepal length (cm)",
     ylab = "Sepal width (cm)")
abline(h = mean(d$Sepal.Width),
       col = "#A88800", lwd = 2)
dev.off()

In [ ]:
IRdisplay::display_png(file = "class07-iris-base.png")

> The gold line is the mean width, not a fitted relationship.

### Draw one panel per species

The grammar of graphics separates data, aesthetic mappings, layers (marks plus a calculation), scales, facets and theme.

In [ ]:
colours <- c("#242424", "#A88800", "#6E6E6E")
p_iris <- ggplot(d,
  aes(Sepal.Length, Sepal.Width,
      colour = Species)) +
  geom_point() +
  geom_hline(yintercept = mean(d$Sepal.Width),
             colour = "#A88800") +
  facet_wrap(~ Species) +
  scale_colour_manual(values = colours) +
  labs(x = "Sepal length (cm)",
       y = "Sepal width (cm)") +
  theme_minimal()
ggsave("class07-iris-facets.pdf", p_iris,
       width = 9, height = 3.5)

In [ ]:
options(repr.plot.width = 9, repr.plot.height = 3.5)
p_iris

> Every panel uses the overall mean as the same gold reference.

## Part 6 · From analysis to a saved figure

### Check eligibility before summarizing

- Use the checked left join.
- Keep records with known hours greater than five and a known positive achievement total.
- Retain the excluded rows for inspection.

In [ ]:
checked <- mutate(joined,
  eligible = !is.na(hours) & hours > 5 &
    !is.na(total) & total > 0 & !is.na(genre))
excluded <- filter(checked, !eligible)
eligible <- filter(checked, eligible)
print(select(excluded, player, game, hours,
             genre, total))
print(c(input = nrow(plays),
        eligible = nrow(eligible),
        excluded = nrow(excluded)))  # 8, 6, 2

> Dan–Portal: hours unknown. Dan–Tunic: no lookup, so no denominator.

### Compute the genre summary

In [ ]:
genre_summary <- eligible |>
  mutate(pct = 100 * achieved / total) |>
  group_by(genre) |>
  summarise(records = n(),
            known_hours = sum(hours),
            pct = mean(pct),
            .groups = "drop") |>
  arrange(desc(pct))
print(genre_summary)

> Each eligible record has equal weight; each genre has two records.

### Save the genre figure

In [ ]:
p_genre <- ggplot(genre_summary,
  aes(reorder(genre, -pct), pct)) +
  geom_col(fill = "#FFD400",
           colour = "#242424") +
  geom_text(aes(label = round(pct, 1)),
            vjust = -0.4) +
  scale_y_continuous(limits = c(0, 50)) +
  labs(x = "Genre",
       y = "Mean achievement completion (%)") +
  theme_minimal()
ggsave("class07-genre-summary.pdf", p_genre,
       width = 7, height = 4)

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4)
p_genre

> puzzle 43.3, platformer 41.7, roguelike 33.7.

### Save the summary and the exclusions

The CSV, the chart and the exclusion record are the inputs for Class B's report.

In [ ]:
write.csv(genre_summary,
          "class07-genre-summary.csv",
          row.names = FALSE)
write.csv(excluded, "class07-excluded.csv",
          row.names = FALSE)

### Recap

- Say what each row represents and how many values a summary uses.
- Keep the original data and a record of exclusions.
- Validate lookup keys; tell unmatched rows from duplicated matches.
- Save a labelled figure with its numerical summary and the code that made both.